<a href="https://colab.research.google.com/github/TanviDudhat/invoice-extraction-llm/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/invoice-extraction-llm")

for sub in ["data/raw", "data/processed", "outputs"]:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
    print("ready:", ROOT / sub)

Mounted at /content/drive
ready: /content/drive/MyDrive/invoice-extraction-llm/data/raw
ready: /content/drive/MyDrive/invoice-extraction-llm/data/processed
ready: /content/drive/MyDrive/invoice-extraction-llm/outputs


In [2]:
import json

SCHEMA = {
    "invoice_number": None,
    "invoice_date": None,
    "vendor_name": None,
    "currency": None,
    "line_items": [
        {"description": None, "quantity": None, "unit_price": None, "amount": None}
    ],
    "subtotal": None,
    "tax": None,
    "total": None,
}

with open(ROOT / "schema.json", "w") as f:
    json.dump(SCHEMA, f, indent=2)

with open(ROOT / "schema.json") as f:
    loaded = json.load(f)

print(json.dumps(loaded, indent=2))

{
  "invoice_number": null,
  "invoice_date": null,
  "vendor_name": null,
  "currency": null,
  "line_items": [
    {
      "description": null,
      "quantity": null,
      "unit_price": null,
      "amount": null
    }
  ],
  "subtotal": null,
  "tax": null,
  "total": null
}


In [3]:
ZIP_URL = "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1"
ZIP_PATH = ROOT / "data/raw/FATURA2.zip"

!wget -q -O "{ZIP_PATH}" "{ZIP_URL}"
!ls -lh "{ZIP_PATH}"

-rw------- 1 root root 659M Nov 23 11:19 /content/drive/MyDrive/invoice-extraction-llm/data/raw/FATURA2.zip


In [4]:
import hashlib

EXPECTED_MD5 = "4c9404462f22c5241eb1a290a02eb2a2"

md5 = hashlib.md5()
with open(ZIP_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        md5.update(chunk)

print("computed:", md5.hexdigest())
print("match:", md5.hexdigest() == EXPECTED_MD5)

computed: 4c9404462f22c5241eb1a290a02eb2a2
match: True


In [5]:
DATA_DIR = Path("/content/fatura")
DATA_DIR.mkdir(exist_ok=True)

!unzip -q -o "{ZIP_PATH}" -d "{DATA_DIR}"
!ls "{DATA_DIR}"

invoices_dataset_final


In [6]:
BASE = DATA_DIR / "invoices_dataset_final"

for item in sorted(BASE.iterdir()):
    if item.is_dir():
        n_files = sum(1 for p in item.rglob("*") if p.is_file())
        print("folder:", item.name, "| files:", n_files)
    else:
        print("file:", item.name)

folder: Annotations | files: 30000
file: Strat2_Split.txt
folder: colored_images | files: 10000
folder: images | files: 10000
file: strat1_dev.csv
file: strat1_test.csv
file: strat1_train.csv


In [7]:
BASE = DATA_DIR / "invoices_dataset_final"

for item in sorted(BASE.iterdir()):
    if item.is_dir():
        n_files = sum(1 for p in item.rglob("*") if p.is_file())
        print("folder:", item.name, "| files:", n_files)
    else:
        print("file:", item.name)

folder: Annotations | files: 30000
file: Strat2_Split.txt
folder: colored_images | files: 10000
folder: images | files: 10000
file: strat1_dev.csv
file: strat1_test.csv
file: strat1_train.csv


In [8]:
BASE = Path("/content/fatura/invoices_dataset_final")

split_path = BASE / "Strat2_Split.txt"
lines = split_path.read_text().splitlines()

print("number of lines:", len(lines))
for line in lines[:12]:
    print(repr(line))

number of lines: 3
'train_inds = [3, 11, 30, 24, 40, 48, 41, 22, 27, 19, 45, 1, 29, 44, 9, 47, 36, 23, 18, 42, 15, 14, 28, 43, 33, 6, 38, 26, 13, 34, 17, 37, 5, 8, 21, 35, 16, 20, 31, 46]'
'dev_inds = [50, 7, 32, 39, 2, 12, 4, 49, 10, 25]'
'test_inds = dev_inds'


In [9]:
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")

print("images:")
for p in sorted((BASE / "images").iterdir())[:5]:
    print("  ", p.name)

print("Annotations:")
for p in sorted((BASE / "Annotations").iterdir())[:5]:
    print("  ", p.name)

images:
   Template10_Instance0.jpg
   Template10_Instance1.jpg
   Template10_Instance10.jpg
   Template10_Instance100.jpg
   Template10_Instance101.jpg
Annotations:
   .virtual_documents
   COCO_compatible_format
   Original_Format
   layoutlm_HF_format


In [10]:
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")

orig = BASE / "Annotations" / "Original_Format"
files = sorted(orig.iterdir())

print("files in Original_Format:", len(files))
print("first three:", [f.name for f in files[:3]])

text = files[0].read_text()
print("characters in first file:", len(text))
print(text[:1500])

files in Original_Format: 10000
first three: ['Template10_Instance0.json', 'Template10_Instance1.json', 'Template10_Instance10.json']
characters in first file: 2167
{"TABLE": [[{"bbox": [[4.0, 458.8898], [534.0, 368.8898]]}]], "INVOICE_INFO": [], "BUYER": {"bbox": [[324.0, 591.4058000000001], [512.712, 675.4058]], "text": "Bill to:Michael Sparks\n4894 Melanie Pass Apt. 742\nTammyland, SD 42587 US\nTel:+(311)738-8211\nEmail:cjames@example.com\nSite:https://www.chase-allen.org/"}, "CONDITIONS": {"bbox": [[26.0, 192.81980000000001], [305.52000000000004, 214.81980000000001]], "text": "Terms and Conditions\nwill be charged if payment is not made within the due date."}, "DATE": {"bbox": [[4.0, 786.4058], [103.38399999999999, 798.4058]], "text": "Date: 03-Jan-1994"}, "DISCOUNT": {"bbox": [[250.0, 324.4058], [407.356, 336.4058]], "text": "DISCOUNT(2.14%): (-)  9.39"}, "DUE_DATE": {"bbox": [[3.0, 733.4058], [135.03599999999997, 745.4058]], "text": "Due Date : 06-Feb-2007"}, "NUMBER": {"bbox": [

In [11]:
import json
from collections import Counter
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

key_counts = Counter()
n_files = 0

for path in orig.glob("*.json"):
    with open(path) as f:
        data = json.load(f)
    key_counts.update(data.keys())
    n_files += 1

print("files read:", n_files)
for key, count in key_counts.most_common():
    print(f"{key:20s} {count}")

files read: 10000
TABLE                10000
INVOICE_INFO         10000
OTHER                10000
LOGO                 10000
DATE                 9800
NUMBER               8800
SELLER_ADDRESS       8200
TOTAL                8000
TITLE                7400
SELLER_NAME          6800
SUB_TOTAL            6797
BUYER                6200
DUE_DATE             5798
NOTE                 5200
SELLER_EMAIL         4400
TOTAL_WORDS          4000
TAX                  3800
PAYMENT_DETAILS      2600
CONDITIONS           2400
DISCOUNT             2400
SEND_TO              1800
BILL_TO              1600
SELLER_SITE          1400
PO_NUMBER            1400
GSTIN                1400
GSTIN_SELLER         1000
AMOUNT_DUE           799
GSTIN_BUYER          600
GST(18%)             600
GST(9%)              400
GST(7%)              200
GST(1%)              200
GST(20%)             200
GST(12%)             200
GST(5%)              200


In [12]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

with_text = 0
example = None

for path in orig.glob("*.json"):
    with open(path) as f:
        data = json.load(f)
    table_str = json.dumps(data["TABLE"])
    if '"text"' in table_str:
        with_text += 1
        if example is None:
            example = (path.name, table_str)

print("files whose TABLE contains text:", with_text)
if example:
    print(example[0])
    print(example[1][:700])

files whose TABLE contains text: 0


In [13]:
import json

SCHEMA_V1 = {
    "invoice_number": None,
    "invoice_date": None,
    "due_date": None,
    "vendor_name": None,
    "subtotal": None,
    "tax": None,
    "total": None,
    "currency": None,
}

with open(ROOT / "schema.json", "w") as f:
    json.dump(SCHEMA_V1, f, indent=2)

print(list(SCHEMA_V1.keys()))

['invoice_number', 'invoice_date', 'due_date', 'vendor_name', 'subtotal', 'tax', 'total', 'currency']


In [14]:
import json
import random
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

paths = sorted(orig.glob("*.json"))
random.seed(0)
sample = random.sample(paths, 400)

totals = []
for path in sample:
    with open(path) as f:
        data = json.load(f)
    text = data.get("TOTAL", {}).get("text")
    if text:
        totals.append(text)

print("invoices with a TOTAL text:", len(totals), "of", len(sample))
for t in totals[:15]:
    print(repr(t))

invoices with a TOTAL text: 320 of 400
'TOTAL : 972.30 EUR'
'TOTAL : 1338.24 USD'
'TOTAL : 780.36 EUR'
'BALANCE DUE : 774.49 EUR'
'TOTAL : 471.63 $'
'TOTAL : 492.81 EUR'
'TOTAL : 585.60 $'
'TOTAL : 573.22 USD'
'TOTAL : 304.36 EUR'
'BALANCE DUE : 138.19 $'
'TOTAL : 606.69 EUR'
'TOTAL : 652.72 $'
'TOTAL : 379.82 $'
'TOTAL : 429.24 EUR'
'TOTAL : 666.02 $'


In [15]:
import re

CURRENCY_MAP = {"$": "USD", "USD": "USD", "EUR": "EUR"}

def parse_total(text):
    match = re.search(r"(\d[\d,]*\.?\d*)\s*(EUR|USD|\$)", text)
    if match is None:
        return None
    amount = float(match.group(1).replace(",", ""))
    currency = CURRENCY_MAP[match.group(2)]
    return amount, currency

results = [parse_total(t) for t in totals]
failures = [t for t, r in zip(totals, results) if r is None]

print(results[:5])
print("failed:", len(failures), "of", len(totals))
for t in failures[:10]:
    print(repr(t))

[(972.3, 'EUR'), (1338.24, 'USD'), (780.36, 'EUR'), (774.49, 'EUR'), (471.63, 'USD')]
failed: 0 of 320


In [16]:
import json

FIELD_KEYS = ["NUMBER", "DATE", "DUE_DATE", "SELLER_NAME", "SUB_TOTAL", "TAX"]

samples = {key: [] for key in FIELD_KEYS}

for path in sample:
    with open(path) as f:
        data = json.load(f)
    for key in FIELD_KEYS:
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        if text and len(samples[key]) < 6:
            samples[key].append(text)

for key, values in samples.items():
    print(key)
    for v in values:
        print("   ", repr(v))

NUMBER
    'INVOICE ID INV/08-29/201'
    'INVOICE # 6608-592'
    'Invoice number INV/90-12/743'
    'INVOICE ID INV61200588'
    'INVOICE # 9005-912'
    'INVOICE ID INV/91-26/667'
DATE
    'Invoice Date: 22-Jun-1998'
    'Invoice Date: 10-Feb-2004'
    'Invoice Date: 15-Mar-2006'
    'Invoice Date: 01-Oct-2007'
    'Invoice Date: 21-Apr-2003'
    'Invoice Date: 07-Jan-2003'
DUE_DATE
    'Due Date : 03-May-2018'
    'Due Date : 24-Mar-2019'
    'Due Date : 05-Nov-2003'
    'Due Date : 01-Feb-1998'
    'Due Date : 23-Mar-2020'
    'Due Date : 11-Mar-2000'
SELLER_NAME
    'Buchanan and Sons'
    'Fernandez PLC'
    'Barnes, Garcia and Martin'
    'Brooks LLC'
    'Morton-Leach'
    'Parker, Phillips and Thomas'
SUB_TOTAL
    'SUB_TOTAL : 1309.36  EUR'
    'SUB_TOTAL : 952.77  EUR'
    'SUB_TOTAL : 764.74  EUR'
    'SUB_TOTAL : 736.62  EUR'
    'SUB_TOTAL : 461.92  $'
    'SUB_TOTAL : 483.96  EUR'
TAX
    'TAX:VAT (4.96%):  65.01 EUR'
    'TAX:VAT (6.38%):  29.48 $'
    'TAX:VAT (6.39%)

In [17]:
import re
from datetime import datetime

CURRENCY_MAP = {"$": "USD", "USD": "USD", "EUR": "EUR"}

def parse_amount(text):
    match = re.search(r"(\d[\d,]*\.?\d*)\s*(EUR|USD|\$)", text)
    if match is None:
        return None
    amount = float(match.group(1).replace(",", ""))
    currency = CURRENCY_MAP[match.group(2)]
    return amount, currency

def parse_number(text):
    pattern = r"^invoice\s*(id|#|number|no\.?)?\s*"
    cleaned = re.sub(pattern, "", text.strip(), flags=re.IGNORECASE)
    return cleaned or None

def parse_date(text):
    match = re.search(r"\d{1,2}-[A-Za-z]{3}-\d{4}", text)
    if match is None:
        return None
    parsed = datetime.strptime(match.group(), "%d-%b-%Y")
    return parsed.date().isoformat()

def parse_name(text):
    return text.strip() or None

tests = [
    ("NUMBER", parse_number),
    ("DATE", parse_date),
    ("DUE_DATE", parse_date),
    ("SELLER_NAME", parse_name),
    ("SUB_TOTAL", parse_amount),
    ("TAX", parse_amount),
]

for key, func in tests:
    print(key)
    for raw in samples[key]:
        print("   ", repr(raw), "->", func(raw))

NUMBER
    'INVOICE ID INV/08-29/201' -> INV/08-29/201
    'INVOICE # 6608-592' -> 6608-592
    'Invoice number INV/90-12/743' -> INV/90-12/743
    'INVOICE ID INV61200588' -> INV61200588
    'INVOICE # 9005-912' -> 9005-912
    'INVOICE ID INV/91-26/667' -> INV/91-26/667
DATE
    'Invoice Date: 22-Jun-1998' -> 1998-06-22
    'Invoice Date: 10-Feb-2004' -> 2004-02-10
    'Invoice Date: 15-Mar-2006' -> 2006-03-15
    'Invoice Date: 01-Oct-2007' -> 2007-10-01
    'Invoice Date: 21-Apr-2003' -> 2003-04-21
    'Invoice Date: 07-Jan-2003' -> 2003-01-07
DUE_DATE
    'Due Date : 03-May-2018' -> 2018-05-03
    'Due Date : 24-Mar-2019' -> 2019-03-24
    'Due Date : 05-Nov-2003' -> 2003-11-05
    'Due Date : 01-Feb-1998' -> 1998-02-01
    'Due Date : 23-Mar-2020' -> 2020-03-23
    'Due Date : 11-Mar-2000' -> 2000-03-11
SELLER_NAME
    'Buchanan and Sons' -> Buchanan and Sons
    'Fernandez PLC' -> Fernandez PLC
    'Barnes, Garcia and Martin' -> Barnes, Garcia and Martin
    'Brooks LLC' -> Broo

In [18]:
%%writefile /content/drive/MyDrive/invoice-extraction-llm/parsers.py
import re
from datetime import datetime

CURRENCY_MAP = {"$": "USD", "USD": "USD", "EUR": "EUR"}

TEXT_FIELDS = [
    ("invoice_number", "NUMBER"),
    ("invoice_date", "DATE"),
    ("due_date", "DUE_DATE"),
    ("vendor_name", "SELLER_NAME"),
]
AMOUNT_FIELDS = [
    ("subtotal", "SUB_TOTAL"),
    ("tax", "TAX"),
    ("total", "TOTAL"),
]


def parse_amount(text):
    match = re.search(r"(\d[\d,]*\.?\d*)\s*(EUR|USD|\$)", text)
    if match is None:
        return None
    amount = float(match.group(1).replace(",", ""))
    currency = CURRENCY_MAP[match.group(2)]
    return amount, currency


def parse_number(text):
    pattern = r"^invoice\s*(id|#|number|no\.?)?\s*"
    cleaned = re.sub(pattern, "", text.strip(), flags=re.IGNORECASE)
    return cleaned or None


def parse_date(text):
    match = re.search(r"\d{1,2}-[A-Za-z]{3}-\d{4}", text)
    if match is None:
        return None
    parsed = datetime.strptime(match.group(), "%d-%b-%Y")
    return parsed.date().isoformat()


def parse_name(text):
    return text.strip() or None


TEXT_PARSERS = {
    "invoice_number": parse_number,
    "invoice_date": parse_date,
    "due_date": parse_date,
    "vendor_name": parse_name,
}


def get_text(data, key):
    value = data.get(key)
    if isinstance(value, dict):
        return value.get("text")
    return None


def build_record(data):
    record = {}
    problems = []
    currencies = []

    for field, key in TEXT_FIELDS:
        text = get_text(data, key)
        value = TEXT_PARSERS[field](text) if text else None
        if text and value is None:
            problems.append(key)
        record[field] = value

    for field, key in AMOUNT_FIELDS:
        text = get_text(data, key)
        parsed = parse_amount(text) if text else None
        if text and parsed is None:
            problems.append(key)
        record[field] = parsed[0] if parsed else None
        if parsed:
            currencies.append(parsed[1])

    if len(set(currencies)) > 1:
        problems.append("CURRENCY_MISMATCH")
    record["currency"] = currencies[0] if currencies else None
    return record, problems

Overwriting /content/drive/MyDrive/invoice-extraction-llm/parsers.py


In [19]:
import re
import sys
import json
import importlib
from pathlib import Path
from collections import Counter

sys.path.append("/content/drive/MyDrive/invoice-extraction-llm")
import parsers
importlib.reload(parsers)

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

records = []
null_counts = Counter()
problem_counts = Counter()
problem_examples = {}

for path in sorted(orig.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    record, problems = parsers.build_record(data)

    m = re.match(r"Template(\d+)_Instance(\d+)", path.stem)
    records.append({"file": path.stem, "template": int(m.group(1)), "target": record})

    for field, value in record.items():
        if value is None:
            null_counts[field] += 1
    problem_counts.update(problems)
    for key in problems:
        problem_examples.setdefault(key, parsers.get_text(data, key))

print("records:", len(records))
print("null counts:", dict(null_counts))
print("problems:", dict(problem_counts))
print("problem examples:", problem_examples)

records: 10000
null counts: {'subtotal': 3203, 'tax': 6200, 'vendor_name': 3200, 'invoice_number': 1200, 'total': 2000, 'due_date': 4202, 'currency': 800, 'invoice_date': 200}
problems: {}
problem examples: {}


In [20]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

gst_with_tax = 0
gst_without_tax = 0
example = None

for path in sorted(orig.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    gst_keys = [k for k in data if k.startswith("GST(")]
    if gst_keys:
        if "TAX" in data:
            gst_with_tax += 1
        else:
            gst_without_tax += 1
            if example is None:
                example = (path.name, {k: data[k] for k in gst_keys})

print("GST key AND TAX key:", gst_with_tax)
print("GST key but NO TAX key:", gst_without_tax)
print(example)

GST key AND TAX key: 200
GST key but NO TAX key: 1000
('Template25_Instance0.json', {'GST(1%)': {'bbox': [[324.0, 87.40576], [411.36, 99.40576]], 'text': 'GST(1%) : 4.63'}, 'GST(20%)': {'bbox': [[322.0, 125.4058], [422.704, 137.4058]], 'text': 'GST(20%) : 92.64'}, 'GST(18%)': {'bbox': [[322.0, 168.4058], [422.704, 180.4058]], 'text': 'GST(18%) : 83.37'}, 'GST(12%)': {'bbox': [[322.0, 205.4058], [422.704, 217.4058]], 'text': 'GST(12%) : 55.58'}, 'GST(5%)': {'bbox': [[322.0, 244.4058], [416.03200000000004, 256.4058]], 'text': 'GST(5%) : 23.16'}})


In [21]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

gst_only_templates = set()

for path in sorted(orig.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    has_gst = any(k.startswith("GST(") for k in data)
    if has_gst and "TAX" not in data:
        number = int(path.stem.split("_")[0].replace("Template", ""))
        gst_only_templates.add(number)

print("GST-only templates:", sorted(gst_only_templates))

with open(orig / "Template25_Instance0.json") as f:
    data = json.load(f)

for key in ["SUB_TOTAL", "TOTAL", "AMOUNT_DUE"]:
    value = data.get(key)
    text = value.get("text") if isinstance(value, dict) else None
    print(key, "->", text)

GST-only templates: [2, 7, 25, 33, 48]
SUB_TOTAL -> SUB_TOTAL : 463.19  EUR
TOTAL -> TOTAL : 476.38 EUR
AMOUNT_DUE -> None


In [22]:
from collections import Counter

checked = 0
ok = 0
bad_by_template = Counter()
examples = []

for r in records:
    t = r["target"]
    if None in (t["subtotal"], t["tax"], t["total"]):
        continue
    checked += 1
    gap = round(t["subtotal"] + t["tax"] - t["total"], 2)
    if abs(gap) <= 0.02:
        ok += 1
    else:
        bad_by_template[r["template"]] += 1
        if len(examples) < 5:
            examples.append((r["file"], t["subtotal"], t["tax"], t["total"], gap))

print("invoices with subtotal, tax and total:", checked)
print("arithmetic holds:", ok)
print("fails by template:", dict(bad_by_template))
for e in examples:
    print(e)

invoices with subtotal, tax and total: 2799
arithmetic holds: 0
fails by template: {11: 200, 12: 200, 17: 200, 18: 200, 1: 200, 28: 200, 29: 200, 31: 200, 43: 200, 44: 200, 46: 199, 50: 200, 5: 200, 6: 200}
('Template11_Instance0', 266.8, 10.26, 264.25, 12.81)
('Template11_Instance1', 502.83, 23.4, 505.92, 20.31)
('Template11_Instance10', 200.96, 9.9, 201.11, 9.75)
('Template11_Instance100', 118.11, 6.83, 122.81, 2.13)
('Template11_Instance101', 713.46, 27.73, 722.43, 18.76)


In [23]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

KEYS = ["SUB_TOTAL", "DISCOUNT", "TAX", "TOTAL", "AMOUNT_DUE"]

for name in ["Template11_Instance0", "Template11_Instance1", "Template11_Instance10"]:
    with open(orig / (name + ".json")) as f:
        data = json.load(f)
    print(name)
    for key in KEYS:
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        print("   ", key, "->", repr(text))

Template11_Instance0
    SUB_TOTAL -> 'SUB_TOTAL : 266.80  $'
    DISCOUNT -> 'DISCOUNT(4.62%): (-)  12.33'
    TAX -> 'TAX:VAT (3.85%):  10.26 $'
    TOTAL -> 'TOTAL : 264.25 $'
    AMOUNT_DUE -> None
Template11_Instance1
    SUB_TOTAL -> 'SUB_TOTAL : 502.83  $'
    DISCOUNT -> 'DISCOUNT(1.59%): (-)  7.99'
    TAX -> 'TAX:VAT (4.65%):  23.40 $'
    TOTAL -> 'TOTAL : 505.92 $'
    AMOUNT_DUE -> None
Template11_Instance10
    SUB_TOTAL -> 'SUB_TOTAL : 200.96  EUR'
    DISCOUNT -> 'DISCOUNT(3.2%): (-)  6.43'
    TAX -> 'TAX:VAT (4.93%):  9.90 EUR'
    TOTAL -> 'TOTAL : 201.11 EUR'
    AMOUNT_DUE -> None


In [24]:
import re
import json
import random
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
OUT = ROOT / "data/processed"
GST_ONLY = {2, 7, 25, 33, 48}

split_text = (BASE / "Strat2_Split.txt").read_text()
train_t = json.loads(re.search(r"train_inds = (\[.*?\])", split_text).group(1))
dev_t = json.loads(re.search(r"dev_inds = (\[.*?\])", split_text).group(1))

shuffled = sorted(dev_t)
random.seed(0)
random.shuffle(shuffled)
new_dev, new_test = set(shuffled[:5]), set(shuffled[5:])

split_of = {t: "train" for t in train_t}
split_of.update({t: "dev" for t in new_dev})
split_of.update({t: "test" for t in new_test})

assert len(split_of) == 50, "a template is in more than one split"

for name in ["train", "dev", "test"]:
    rows = [{**r, "gst_rate_table": r["template"] in GST_ONLY}
            for r in records if split_of[r["template"]] == name]
    with open(OUT / (name + ".jsonl"), "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    templates = sorted({r["template"] for r in rows})
    print(name, len(rows), "invoices | templates:", templates)

train 8000 invoices | templates: [1, 3, 5, 6, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48]
dev 1000 invoices | templates: [4, 10, 25, 39, 49]
test 1000 invoices | templates: [2, 7, 12, 32, 50]


In [25]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"


def collect_boxes(obj):
    found = []
    if isinstance(obj, dict):
        if "text" in obj:
            found.append(obj)
        else:
            for value in obj.values():
                found.extend(collect_boxes(value))
    elif isinstance(obj, list):
        for item in obj:
            found.extend(collect_boxes(item))
    return found


def build_input(data):
    boxes = collect_boxes(data)

    def position(box):
        coords = box.get("bbox")
        if not coords:
            return (float("-inf"), 0)
        (x1, y1), (x2, y2) = coords
        return (-max(y1, y2), min(x1, x2))

    boxes.sort(key=position)
    return "\n".join(box["text"] for box in boxes)


with open(orig / "Template11_Instance0.json") as f:
    data = json.load(f)

text = build_input(data)
print(len(text), "characters")
print(text)

803 characters
TAX INVOICE

Invoice number INV/72-06/714
Invoice Date: 20-Apr-2010
Due Date : 25-Jul-2015

Property range. $17.53 5.00 $87.65
Accept drug. $45.24 2.00 $90.48
Guess western scientist. $31.55 1.00 $31.55
Home ioin reach him. $28.56 2.00 $57.12

DISCOUNT(4.62%): (-) 12.33
SUB_TOTAL : 266.80 $
TAX:VAT (3.85%): 10.26 $

TOTAL : 264.25 $

Terms and Conditions
will be charged if payment is not made within the due date.

Note:Total payment due in 14 days.

Email:lisa97@example.org

TAX INVOICE
Invoice number INV/72-06/714
Invoice Date: 20-Apr-2010
Due Date : 25-Jul-2015
DISCOUNT(4.62%): (-)  12.33
SUB_TOTAL : 266.80  $
TAX:VAT (3.85%):  10.26 $
TOTAL : 264.25 $
Terms and Conditions
will be charged if payment is not made within the due date.
Note:Total payment due in 14 days.
Email:lisa97@example.org


In [26]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"


def norm(s):
    return " ".join(s.split())


with open(orig / "Template11_Instance0.json") as f:
    data = json.load(f)

other = data["OTHER"]
print("OTHER is a:", type(other).__name__)
other_text = other.get("text") if isinstance(other, dict) else None
print("OTHER has text:", other_text is not None)

if other_text:
    print(len(other_text), "characters")
    print(repr(other_text[:300]))
    haystack = norm(other_text)
    for key in ["NUMBER", "DATE", "SUB_TOTAL", "TAX", "TOTAL"]:
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        if text:
            print(key, "inside OTHER:", norm(text) in haystack)

OTHER is a: dict
OTHER has text: True
479 characters
'TAX INVOICE\n\nInvoice number INV/72-06/714\nInvoice Date: 20-Apr-2010\nDue Date : 25-Jul-2015\n\nProperty range. $17.53 5.00 $87.65\nAccept drug. $45.24 2.00 $90.48\nGuess western scientist. $31.55 1.00 $31.55\nHome ioin reach him. $28.56 2.00 $57.12\n\nDISCOUNT(4.62%): (-) 12.33\nSUB_TOTAL : 266.80 $\nTAX:VAT '
NUMBER inside OTHER: True
DATE inside OTHER: True
SUB_TOTAL inside OTHER: True
TAX inside OTHER: True
TOTAL inside OTHER: True


In [27]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

FIELD_KEYS = ["NUMBER", "DATE", "DUE_DATE", "SELLER_NAME", "SUB_TOTAL", "TAX", "TOTAL"]


def norm(s):
    return " ".join(s.split())


no_other = Counter()
miss_by_key = Counter()
lengths = []
example = None

for path in sorted(orig.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    template = int(path.stem.split("_")[0].replace("Template", ""))

    other = data.get("OTHER")
    other_text = other.get("text") if isinstance(other, dict) else None
    if not other_text:
        no_other[template] += 1
        continue

    lengths.append(len(other_text))
    haystack = norm(other_text)
    for key in FIELD_KEYS:
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        if text and norm(text) not in haystack:
            miss_by_key[key] += 1
            if example is None:
                example = (path.name, key, text)

print("files with OTHER text:", len(lengths))
print("files without it, by template:", dict(no_other))
print("fields NOT found inside OTHER:", dict(miss_by_key))
if lengths:
    print("OTHER length min/avg/max:", min(lengths), sum(lengths) // len(lengths), max(lengths))
print("first miss:", example)

files with OTHER text: 10000
files without it, by template: {}
fields NOT found inside OTHER: {'SELLER_NAME': 6800, 'NUMBER': 130, 'TOTAL': 8, 'DATE': 128, 'SUB_TOTAL': 290, 'TAX': 215, 'DUE_DATE': 3}
OTHER length min/avg/max: 176 604 1141
first miss: ('Template10_Instance0.json', 'SELLER_NAME', 'Mclean-Cochran')


In [28]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

FIELD_KEYS = ["NUMBER", "DATE", "DUE_DATE", "SUB_TOTAL", "TAX", "TOTAL"]


def norm(s):
    return " ".join(s.split())


shown = set()

for path in sorted(orig.glob("*.json")):
    if len(shown) == len(FIELD_KEYS):
        break
    with open(path) as f:
        data = json.load(f)
    other = data.get("OTHER")
    other_text = other.get("text") if isinstance(other, dict) else None
    if not other_text:
        continue
    haystack = norm(other_text)
    for key in FIELD_KEYS:
        if key in shown:
            continue
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        if text and norm(text) not in haystack:
            shown.add(key)
            print(path.name, key)
            print("  labeled:", repr(text))
            print("  OTHER  :", repr(other_text))

Template10_Instance101.json NUMBER
  labeled: 'Invoice number INV07820771'
  OTHER  : 'Address:4894 Melanie Pass Apt. 742\nTammyland, SD 42587 US\n\nInvoice Date: 23-Dec-2015\n\nInvoice number INVO7820771\nSK DIGITAL\n\nDue Date : 06-Jul-1998\n\nBuyer :Valerie Huffman\n\n233 Robert Cove Apt. 097\nJohnsonfurt, MS 39320 US\nTel:+(366)483-4826\nEmail:qyoung@example.org\nSite:https://solomon.com/\n\nITEMS QUANTITY PRICE TAX AMOUNT\nMiddle teach. 1.00 $22.02 $0.00 $22.02\nAway situation. 3.00 $33.31 $0.00 $99.93\nThis firm. 3.00 $8.91 $0.00 $26.73\nAuthor student. 1.00 $4.83 $0.00 $4.83\n\nDISCOUNT(1.11%): (-) 1.7\n\nTOTAL : 156.25 EUR\n\nTerms and Conditions\nwill be charged if payment is not made within the due date.\n\nThank you for your business!\n\nEmail:cjames@example.com\n'
Template11_Instance1.json TOTAL
  labeled: 'TOTAL : 505.92 $'
  OTHER  : 'INVOICE\n\nINVOICE # 0875-302\n\nInvoice Date: 30-Nov-2021\n\nDue Date : 17-Feb-2008\n\nem i) a\nBy air. $83.90\nProtect month media. $74.4

In [29]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"

FIELD_KEYS = ["NUMBER", "DATE", "DUE_DATE", "SUB_TOTAL", "TAX", "TOTAL"]


def norm(s):
    return " ".join(s.split())


shown = set()

for path in sorted(orig.glob("*.json")):
    if len(shown) == len(FIELD_KEYS):
        break
    with open(path) as f:
        data = json.load(f)
    other = data.get("OTHER")
    other_text = other.get("text") if isinstance(other, dict) else None
    if not other_text:
        continue
    haystack = norm(other_text)
    for key in FIELD_KEYS:
        if key in shown:
            continue
        value = data.get(key)
        text = value.get("text") if isinstance(value, dict) else None
        if text and norm(text) not in haystack:
            shown.add(key)
            print(path.name, key)
            print("  labeled:", repr(text))
            print("  OTHER  :", repr(other_text))

Template10_Instance101.json NUMBER
  labeled: 'Invoice number INV07820771'
  OTHER  : 'Address:4894 Melanie Pass Apt. 742\nTammyland, SD 42587 US\n\nInvoice Date: 23-Dec-2015\n\nInvoice number INVO7820771\nSK DIGITAL\n\nDue Date : 06-Jul-1998\n\nBuyer :Valerie Huffman\n\n233 Robert Cove Apt. 097\nJohnsonfurt, MS 39320 US\nTel:+(366)483-4826\nEmail:qyoung@example.org\nSite:https://solomon.com/\n\nITEMS QUANTITY PRICE TAX AMOUNT\nMiddle teach. 1.00 $22.02 $0.00 $22.02\nAway situation. 3.00 $33.31 $0.00 $99.93\nThis firm. 3.00 $8.91 $0.00 $26.73\nAuthor student. 1.00 $4.83 $0.00 $4.83\n\nDISCOUNT(1.11%): (-) 1.7\n\nTOTAL : 156.25 EUR\n\nTerms and Conditions\nwill be charged if payment is not made within the due date.\n\nThank you for your business!\n\nEmail:cjames@example.com\n'
Template11_Instance1.json TOTAL
  labeled: 'TOTAL : 505.92 $'
  OTHER  : 'INVOICE\n\nINVOICE # 0875-302\n\nInvoice Date: 30-Nov-2021\n\nDue Date : 17-Feb-2008\n\nem i) a\nBy air. $83.90\nProtect month media. $74.4

In [30]:
import json
from pathlib import Path

BASE = Path("/content/fatura/invoices_dataset_final")
orig = BASE / "Annotations" / "Original_Format"
OUT = ROOT / "data/processed"

CHECKS = [
    ("invoice_number", "NUMBER"),
    ("invoice_date", "DATE"),
    ("due_date", "DUE_DATE"),
    ("subtotal", "SUB_TOTAL"),
    ("tax", "TAX"),
    ("total", "TOTAL"),
]


def norm(s):
    return " ".join(s.split())


for name in ["train", "dev", "test"]:
    with open(OUT / (name + ".jsonl")) as f:
        rows = [json.loads(line) for line in f]

    unsupported = {field: 0 for field, _ in CHECKS}

    for row in rows:
        with open(orig / (row["file"] + ".json")) as g:
            data = json.load(g)
        other = data.get("OTHER")
        text = other.get("text", "") if isinstance(other, dict) else ""
        row["input"] = text
        haystack = norm(text)

        supported = {}
        for field, key in CHECKS:
            value = data.get(key)
            raw = value.get("text") if isinstance(value, dict) else None
            if raw:
                supported[field] = norm(raw) in haystack
                if not supported[field]:
                    unsupported[field] += 1
        row["supported"] = supported

    with open(OUT / (name + ".jsonl"), "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    print(name, len(rows), "rows | unsupported:", unsupported)

train 8000 rows | unsupported: {'invoice_number': 113, 'invoice_date': 123, 'due_date': 0, 'subtotal': 89, 'tax': 32, 'total': 4}
dev 1000 rows | unsupported: {'invoice_number': 8, 'invoice_date': 0, 'due_date': 1, 'subtotal': 0, 'tax': 0, 'total': 4}
test 1000 rows | unsupported: {'invoice_number': 9, 'invoice_date': 5, 'due_date': 2, 'subtotal': 201, 'tax': 183, 'total': 0}


In [31]:
import json
from collections import Counter

OUT = ROOT / "data/processed"


def load(name):
    with open(OUT / (name + ".jsonl")) as f:
        return [json.loads(line) for line in f]


def norm(s):
    return " ".join(s.split())


data = {name: load(name) for name in ["train", "dev", "test"]}

bad = Counter()
for row in data["test"]:
    for field in ["subtotal", "tax"]:
        if row["supported"].get(field) is False:
            bad[(row["template"], field)] += 1
print("test noise by (template, field):", dict(bad))

inputs = {name: {norm(r["input"]) for r in rows} for name, rows in data.items()}
print("unique inputs:", {name: len(s) for name, s in inputs.items()})
print("train/test overlap:", len(inputs["train"] & inputs["test"]))
print("train/dev overlap:", len(inputs["train"] & inputs["dev"]))
print("dev/test overlap:", len(inputs["dev"] & inputs["test"]))

test noise by (template, field): {(12, 'subtotal'): 176, (12, 'tax'): 182, (32, 'subtotal'): 2, (50, 'subtotal'): 23, (50, 'tax'): 1}
unique inputs: {'train': 8000, 'dev': 1000, 'test': 1000}
train/test overlap: 0
train/dev overlap: 0
dev/test overlap: 0


In [ ]:
import json
from transformers import AutoTokenizer

OUT = ROOT / "data/processed"
FIELDS = ["invoice_number", "invoice_date", "due_date",
          "subtotal", "tax", "total", "currency"]


def make_prompt(invoice_text):
    return "Extract the invoice fields as JSON.\n\nInvoice:\n" + invoice_text.strip() + "\n\nJSON:"


def make_completion(target):
    clean = {key: target[key] for key in FIELDS}
    return " " + json.dumps(clean)


tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

for name in ["train", "dev", "test"]:
    with open(OUT / (name + ".jsonl")) as f:
        rows = [json.loads(line) for line in f]

    if name == "train":
        rows = [r for r in rows if all(r["supported"].values())]

    sft_rows = [
        {
            "file": r["file"],
            "template": r["template"],
            "prompt": make_prompt(r["input"]),
            "completion": make_completion(r["target"]),
            "gst_rate_table": r["gst_rate_table"],
            "supported": r["supported"],
        }
        for r in rows
    ]

    with open(OUT / ("sft_" + name + ".jsonl"), "w") as f:
        for row in sft_rows:
            f.write(json.dumps(row) + "\n")

    lengths = sorted(len(tok(r["prompt"] + r["completion"])["input_ids"]) for r in sft_rows)
    print(name, len(sft_rows), "rows | tokens avg / 95th pct / max:",
          sum(lengths) // len(lengths), lengths[int(0.95 * len(lengths))], lengths[-1])

print(sft_rows[0]["prompt"])
print(repr(sft_rows[0]["completion"]))